# ClinVision Kaggle Notebook

Purpose: manually validate a future approved public-data-only ClinVision run with licence-cleared paired image/report data.

Safety rules: do not use MIMIC-CXR, restricted clinical data, private reports, patient uploads, provider APIs, Hugging Face uploads, W&B, deployment, model publication, or clinical-facing claims without separate approval.

Expected outputs: sanitized environment inventory, input inventory, copied source tree, provider-free synthetic validation, preserved evidence inventory, and a final blocker summary.

In [ ]:
from pathlib import Path
import json
import os
import platform
import sys

summary = {
    "python": sys.version,
    "platform": platform.platform(),
    "cwd": os.getcwd(),
    "kaggle_input_exists": Path("/kaggle/input").exists(),
    "kaggle_working_exists": Path("/kaggle/working").exists(),
}
print(json.dumps(summary, indent=2))

In [ ]:
from pathlib import Path
import json

input_root = Path("/kaggle/input")
items = []
if input_root.exists():
    for path in sorted(input_root.iterdir()):
        items.append({"name": path.name, "is_dir": path.is_dir()})
print(json.dumps({"input_root": str(input_root), "items": items}, indent=2))
print("Do not print raw report text, image pixels, identifiers, or file contents in this inspection cell.")

In [ ]:
from pathlib import Path
import shutil

source_candidates = [
    Path("/kaggle/input/clinvision-source"),
    Path("/kaggle/input/13-clinvision"),
]
source_root = next((path for path in source_candidates if path.exists()), None)
if source_root is None:
    raise RuntimeError("Attach an approved ClinVision source snapshot under /kaggle/input before running this cell.")

target_root = Path("/kaggle/working/clinvision")
if target_root.exists():
    raise RuntimeError("Target already exists. Inspect it manually before replacing anything.")
ignore = shutil.ignore_patterns(".git", ".pytest_cache", "__pycache__", "*.pyc", "data", "datasets", "weights", "checkpoints", "outputs")
shutil.copytree(source_root, target_root, ignore=ignore)
print({"copied_from": str(source_root), "copied_to": str(target_root)})

In [ ]:
raise RuntimeError("Approval gate: edit this cell only inside Kaggle after dependency versions and install commands are approved. Restart the kernel after installation.")
# Example shape after approval only, one command at a time:
# %pip install --no-cache-dir -r /kaggle/working/clinvision/requirements-kaggle.txt

## Kernel Restart Gate

If dependencies were installed in the previous cell, restart the Kaggle kernel before continuing. After restart, rerun cells 01, 02, and any approved setup cells needed to restore local variables.

In [ ]:
from pathlib import Path
import subprocess
import sys

project_root = Path("/kaggle/working/clinvision")
if not project_root.exists():
    raise RuntimeError("Copy the source tree before running synthetic validation.")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_contracts.py", "-q"],
    cwd=project_root,
    text=True,
    capture_output=True,
    check=False,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
raise SystemExit(result.returncode)

In [ ]:
from pathlib import Path
import json

project_root = Path("/kaggle/working/clinvision")
evidence_dirs = [project_root / "docs", project_root / "notebooks", project_root / "artifacts" / "synthetic"]
evidence = []
for folder in evidence_dirs:
    if folder.exists():
        for path in sorted(folder.rglob("*")):
            if path.is_file() and path.suffix.lower() in {".md", ".json", ".ipynb", ".txt"}:
                evidence.append(str(path.relative_to(project_root)))
print(json.dumps({"evidence_files": evidence}, indent=2))
print("Preserve historical evidence. Do not delete or overwrite evidence files without review.")

## Approval Gate Before Real Work

Stop here unless there is explicit approval for the exact public dataset, item-level licence ledger, model/runtime versions, model downloads, providers, GPU or heavy CPU work, artifact handling, and paste-back fields. MIMIC-CXR remains blocked.

In [ ]:
raise RuntimeError("Optional approved command only: validate the reviewed item-level public-data licence ledger.")

In [ ]:
raise RuntimeError("Optional approved command only: build a non-identifying manifest from approved public data.")

In [ ]:
raise RuntimeError("Optional approved command only: run BLIP-2 generation boundary checks after model/runtime approval.")

In [ ]:
raise RuntimeError("Optional approved command only: run BiomedCLIP retrieval boundary checks after model/runtime approval.")

In [ ]:
raise RuntimeError("Optional approved command only: compute non-clinical text metrics after approved data/model outputs exist.")

In [ ]:
from pathlib import Path
import json

project_root = Path("/kaggle/working/clinvision")
summary = {
    "project_root_exists": project_root.exists(),
    "release_default": "blocked",
    "remaining_blockers": [
        "real public dataset licence audit",
        "model/runtime approval",
        "GPU or heavy CPU approval",
        "privacy review on real data",
        "release/publication approval",
    ],
}
print(json.dumps(summary, indent=2))
raise SystemExit("Stop after preserving sanitized evidence. Do not publish or deploy from this notebook.")